# QMS Wait-Time ML Pipeline

**Purpose**: Train and validate a HistGradientBoosting model to predict queue waiting time for Malaysian bank branches.

**Pipeline Lifecycle**:
1. Data Ingestion (synthetic + optional external datasets)
2. Data Cleansing (abandoned tickets, outliers, dedup)
3. Feature Engineering (24 features aligned with production)
4. Model Training (temporal split, sample weighting, Poisson loss)
5. Evaluation & Comparison (ML vs Formula baseline)

**System**: QMS (Queue Management System) — online booking + walk-in, multi-counter, CreatedAt FIFO call ordering

## 0. Setup

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import cross_val_score, TimeSeriesSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OrdinalEncoder

print('Libraries loaded.')

## 1. Data Ingestion

Generate synthetic data that matches production system behavior:
- **CreatedAt FIFO** call ordering (matches `QueueCallNextSelector`)
- **Multi-counter** drain with service assignments (matches `CounterSimulationEstimator`)
- **24 features** aligned with `MlHttpWaitTimeEstimator.BuildPayload`
- **Target** = `ServingStartedAt - QueueEligibleAt` (matches `MlTrainingObservation`)

In [ ]:
# ── Branch & Service configs (matching production) ──

BRANCHES = [
    {"BranchCode": 101033, "base_traffic": 1.0, "slot_duration": 30, "early_call": 15},
    {"BranchCode": 101045, "base_traffic": 0.8, "slot_duration": 30, "early_call": 15},
    {"BranchCode": 101012, "base_traffic": 1.3, "slot_duration": 30, "early_call": 10},
    {"BranchCode": 101078, "base_traffic": 0.9, "slot_duration": 30, "early_call": 15},
    {"BranchCode": 101056, "base_traffic": 1.1, "slot_duration": 30, "early_call": 15},
]

SERVICES = [
    {"code": "DEPOSIT",     "prefix": "A", "avg_min": 6,  "online_slots": 4, "weight": 0.16},
    {"code": "WITHDRAW",    "prefix": "A", "avg_min": 5,  "online_slots": 4, "weight": 0.06},
    {"code": "LOAN",        "prefix": "B", "avg_min": 30, "online_slots": 2, "weight": 0.08},
    {"code": "CREDIT_CARD", "prefix": "C", "avg_min": 12, "online_slots": 3, "weight": 0.10},
    {"code": "ACC_OPEN",    "prefix": "D", "avg_min": 25, "online_slots": 2, "weight": 0.08},
    {"code": "CUST_SVC",    "prefix": "E", "avg_min": 10, "online_slots": 5, "weight": 0.14},
    {"code": "ATM_CDM",     "prefix": "F", "avg_min": 8,  "online_slots": 3, "weight": 0.10},
    {"code": "FIXED_DEP",   "prefix": "G", "avg_min": 15, "online_slots": 2, "weight": 0.06},
    {"code": "INSURANCE",   "prefix": "H", "avg_min": 20, "online_slots": 2, "weight": 0.06},
    {"code": "BIZ_BANK",    "prefix": "I", "avg_min": 18, "online_slots": 2, "weight": 0.06},
    {"code": "ONLINE_SUP",  "prefix": "J", "avg_min": 12, "online_slots": 3, "weight": 0.05},
    {"code": "REMITTANCE",  "prefix": "K", "avg_min": 10, "online_slots": 3, "weight": 0.05},
]

PROFILES = [
    ["A", "E"], ["A", "F"], ["B", "C"],
    ["D", "G", "H"], ["I", "J", "K"], ["A", "B", "C", "E"],
]

print(f'{len(BRANCHES)} branches, {len(SERVICES)} services, {len(PROFILES)} counter profiles')

In [ ]:
from dataclasses import dataclass

@dataclass
class Ticket:
    ticket_id: int
    branch_code: int
    service_code: str
    prefix: str
    default_avg_min: float
    online_slots_per_slot: int
    entry_type: int   # 0=Online, 1=WalkIn
    checked_in: bool
    created_at: float  # minutes from midnight
    slot_start: float = None
    slot_end: float = None
    queue_eligible_at: float = 0.0
    serving_started_at: float = None
    service_end_at: float = None      # actual end time (observable: counter becomes free)
    abandoned: bool = False
    enqueue_seq: int = 0

@dataclass
class Counter:
    counter_id: int
    allowed_prefixes: list
    free_at: float = 0.0

def hour_traffic(h): return {9:1.4,10:1.6,11:1.3,12:0.7,13:0.8,14:1.3,15:1.5,16:1.2}.get(h,0.5)
def day_traffic(d):  return {0:1.4,1:1.1,2:1.0,3:1.0,4:1.3}.get(d,0.0)

print('Simulation classes defined.')

In [ ]:
def simulate_day(rng, branch, day_of_week, n_profiles):
    """Discrete-event simulation of one day at one branch."""
    svc_codes = [s['code'] for s in SERVICES]
    svc_weights = np.array([s['weight'] for s in SERVICES], dtype=float)
    svc_weights /= svc_weights.sum()
    svc_by_code = {s['code']: s for s in SERVICES}
    slot_dur = branch['slot_duration']
    early_call = branch['early_call']

    # Set up counters
    profile_indices = rng.choice(len(PROFILES), size=n_profiles, replace=False)
    counters = [Counter(i, PROFILES[idx]) for i, idx in enumerate(profile_indices)]

    day_mult = day_traffic(day_of_week)
    if day_mult == 0: return []

    # Generate arrivals
    tickets, tid = [], 0
    completed_durations = {s['prefix']: [s['avg_min'] * rng.uniform(0.6,1.4) for _ in range(10)] for s in SERVICES}

    for hour in range(9, 17):
        lam = branch['base_traffic'] * hour_traffic(hour) * day_mult * 8
        for _ in range(max(0, rng.poisson(lam))):
            created_at = hour * 60 + rng.uniform(0, 60)
            svc = svc_by_code[str(rng.choice(svc_codes, p=svc_weights))]
            entry_type = 0 if rng.random() < 0.35 else 1
            slot_start = slot_end = None
            queue_eligible_at = created_at

            if entry_type == 0:  # Online
                actual_created = created_at - rng.exponential(4) * 60
                slot_start = (int(created_at) // slot_dur) * slot_dur
                slot_end = slot_start + slot_dur
                queue_eligible_at = slot_start
                created_at = actual_created
                checked_in = rng.random() < 0.88
                abandoned = not checked_in and rng.random() < 0.6
            else:
                checked_in = True
                abandoned = rng.random() < 0.05

            tid += 1
            tickets.append(Ticket(tid, branch['BranchCode'], svc['code'], svc['prefix'],
                                  svc['avg_min'], svc['online_slots'], entry_type, checked_in,
                                  created_at, slot_start, slot_end, queue_eligible_at, None, None, abandoned, tid))

    if not tickets: return []
    waiting = [t for t in tickets if not t.abandoned]

    def is_eligible(t, now):
        if t.serving_started_at is not None or t.abandoned: return False
        if t.entry_type == 1: return t.checked_in and t.created_at <= now
        if not t.checked_in or t.slot_start is None: return False
        return (t.slot_start - early_call) <= now

    def rolling_avg(prefix):
        d = completed_durations[prefix][-30:]
        return float(np.mean(d)) if d else 10.0

    # Discrete-event Call Next drain (CreatedAt FIFO)
    for c in counters: c.free_at = 540.0
    for _ in range(len(waiting) * 3 + 1000):
        ec = min(counters, key=lambda c: c.free_at)
        now = ec.free_at
        if now > 1050: break

        pool = [t for t in waiting if is_eligible(t, now) and t.serving_started_at is None
                and t.prefix in ec.allowed_prefixes]
        if not pool:
            next_t = 1050
            for t in waiting:
                if t.serving_started_at is None and not t.abandoned and t.entry_type == 0 and t.checked_in and t.slot_start:
                    et = t.slot_start - early_call
                    if et > now: next_t = min(next_t, et)
            if next_t >= 1050:
                if all(c.free_at <= now for c in counters): break
                ec.free_at = now + 1; continue
            ec.free_at = next_t; continue

        pool.sort(key=lambda t: (t.created_at, t.ticket_id))
        ticket = pool[0]
        svc_dur = ticket.default_avg_min * rng.uniform(0.5, 1.8)
        if rng.random() < 0.03: svc_dur *= rng.uniform(2, 4)
        ticket.serving_started_at = now
        ticket.service_end_at = now + svc_dur   # record actual end time
        ec.free_at = now + svc_dur
        completed_durations[ticket.prefix].append(svc_dur)

    # Build snapshot rows for served tickets
    rows = []
    for t in [t for t in tickets if t.serving_started_at is not None]:
        snap = t.queue_eligible_at
        hour = max(0, min(23, int(snap // 60)))
        is_peak = 1 if (9<=hour<=11 or 14<=hour<=16) else 0

        same_svc = [o for o in tickets if o.service_code == t.service_code and not o.abandoned
                    and o.queue_eligible_at <= snap and (o.serving_started_at is None or o.serving_started_at > snap)]
        people_ahead = sum(1 for o in same_svc if is_eligible(o, snap)
                          and (o.created_at < t.created_at or (o.created_at == t.created_at and o.ticket_id < t.ticket_id)))
        # FIX: Use actual service_end_at instead of default_avg_min * 1.2 estimate.
        # In production, "is counter busy?" is directly observable (person at window).
        # Using default_avg_min * 1.2 was a subtle data leak — peeking at statistical
        # distribution to guess service completion, which the system can't know at snap time.
        now_serving = sum(1 for o in tickets if o.service_code == t.service_code
                         and o.serving_started_at is not None and o.service_end_at is not None
                         and o.serving_started_at <= snap and o.service_end_at > snap)

        heard = set()
        for c in counters:
            if t.prefix in c.allowed_prefixes: heard.update(c.allowed_prefixes)
        cross_lane = sum(1 for o in tickets if o.prefix in heard and not o.abandoned
                         and o.queue_eligible_at <= snap and (o.serving_started_at is None or o.serving_started_at > snap))

        slot_active = 1 if t.slot_start and t.slot_end and t.slot_start <= snap < t.slot_end else 0
        rows.append({
            'QueueLength': len(same_svc), 'CrossLaneQueueLength': cross_lane,
            'PeopleAheadCallNext': people_ahead, 'NowServing': now_serving,
            'ActiveCounters': len(counters),
            'ListeningCounters': sum(1 for c in counters if t.prefix in c.allowed_prefixes),
            'HourOfDay': hour, 'DayOfWeek': day_of_week, 'IsPeakHour': is_peak,
            'DefaultAvgServiceMinutes': t.default_avg_min,
            'EnqueueSequence': t.enqueue_seq, 'EntryType': t.entry_type,
            'CheckedIn': 1 if t.checked_in else 0,
            'SlotActive': slot_active, 'CallNextPriority': 0 if (t.entry_type==0 and t.checked_in and slot_active) else 1,
            'SlotDurationMinutes': slot_dur,
            'MinutesUntilSlotEnd': max(0, t.slot_end - snap) if t.slot_end else None,
            'MinutesSinceSlotStart': max(0, snap - t.slot_start) if t.slot_start else None,
            'OnlineSlotsPerSlot': t.online_slots_per_slot,
            'OnlineBookedInSlot': sum(1 for o in tickets if o.service_code==t.service_code and o.slot_start==t.slot_start and o.entry_type==0 and not o.abandoned) if t.slot_start else 0,
            'WalkInInSlot': sum(1 for o in tickets if o.service_code==t.service_code and o.slot_start==t.slot_start and o.entry_type==1 and not o.abandoned) if t.slot_start else 0,
            'BranchCode': t.branch_code, 'ServiceCode': t.service_code, 'TicketPrefix': t.prefix,
            'WaitingMinutes': round(t.serving_started_at - t.queue_eligible_at, 2),
        })

    # Also emit abandoned tickets (WaitingMinutes=None, cleaned in Stage 2)
    for t in tickets:
        if t.abandoned:
            rows.append({
                'QueueLength': 0, 'CrossLaneQueueLength': 0,
                'PeopleAheadCallNext': 0, 'NowServing': 0,
                'ActiveCounters': len(counters),
                'ListeningCounters': sum(1 for c in counters if t.prefix in c.allowed_prefixes),
                'HourOfDay': int(t.created_at // 60), 'DayOfWeek': day_of_week, 'IsPeakHour': 0,
                'DefaultAvgServiceMinutes': t.default_avg_min,
                'EnqueueSequence': t.enqueue_seq, 'EntryType': t.entry_type,
                'CheckedIn': 0,
                'SlotActive': 0, 'CallNextPriority': 1,
                'SlotDurationMinutes': slot_dur,
                'MinutesUntilSlotEnd': None, 'MinutesSinceSlotStart': None,
                'OnlineSlotsPerSlot': t.online_slots_per_slot,
                'OnlineBookedInSlot': 0, 'WalkInInSlot': 0,
                'BranchCode': t.branch_code, 'ServiceCode': t.service_code, 'TicketPrefix': t.prefix,
                'WaitingMinutes': None,  # abandoned — no service
            })

    return rows

print('Simulation function defined.')

In [ ]:
%%time
# Generate 30K rows across multiple days and branches
rng = np.random.default_rng(42)
all_rows = []
day_count = 0
TARGET_ROWS = 30000

while len(all_rows) < TARGET_ROWS:
    for br in BRANCHES:
        if len(all_rows) >= TARGET_ROWS: break
        dow = day_count % 5
        n_prof = min(len(PROFILES), max(2, int(rng.integers(3, 7))))
        all_rows.extend(simulate_day(rng, br, dow, n_prof))
    day_count += 1

df = pd.DataFrame(all_rows)

# Winsorize at p99
served = df['WaitingMinutes'].notna()
if served.sum() > 0:
    p99 = df.loc[served, 'WaitingMinutes'].quantile(0.99)
    df.loc[served, 'WaitingMinutes'] = df.loc[served, 'WaitingMinutes'].clip(upper=p99)

df['DataSource'] = 'synthetic'
print(f'Generated {len(df)} rows ({served.sum()} served)')
print(f'Wait time: mean={df["WaitingMinutes"].mean():.1f}, median={df["WaitingMinutes"].median():.1f}, p95={df["WaitingMinutes"].quantile(0.95):.1f}')

In [ ]:
# Visualize synthetic data distribution
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

s = df[df['WaitingMinutes'].notna()]
axes[0].hist(s['WaitingMinutes'], bins=60, color='steelblue', edgecolor='white')
axes[0].set_xlabel('Wait Time (min)')
axes[0].set_title(f'Wait Time Distribution (n={len(s):,})')
axes[0].axvline(s['WaitingMinutes'].median(), color='red', ls='--', label=f'median={s["WaitingMinutes"].median():.1f}')
axes[0].legend()

axes[1].bar(range(9,17), [s[s['HourOfDay']==h]['WaitingMinutes'].mean() for h in range(9,17)], color='coral')
axes[1].set_xlabel('Hour')
axes[1].set_ylabel('Avg Wait (min)')
axes[1].set_title('Wait Time by Hour')

svc_avg = s.groupby('ServiceCode')['WaitingMinutes'].mean().sort_values(ascending=False)
axes[2].barh(svc_avg.index, svc_avg.values, color='mediumseagreen')
axes[2].set_xlabel('Avg Wait (min)')
axes[2].set_title('Wait Time by Service')

plt.tight_layout()
plt.show()

### 1b. Combine External Datasets (Optional)

Upload `hospital_engineered.csv` and `college_engineered.csv` to Colab, then run the cell below.
If you skip this cell, the pipeline trains on **synthetic data only** (recommended for launch).

Three safeguards prevent external data from hurting the model:
1. **Filter**: Hospital keeps only `active_counters ≥ 3` (drops 1/2-counter disaster scenarios)
2. **Scale normalization**: Align external median wait to synthetic median
3. **Sample weighting**: External gets weight 0.1 vs synthetic 1.0 during training

In [ ]:
# ── Upload external datasets (Colab only) ──
# Option 1: Drag files into the left sidebar Files panel (easiest)
# Option 2: Run this cell — it will only upload files that are missing

import os

needed = ['hospital_engineered.csv', 'college_engineered.csv']
missing = [f for f in needed if not os.path.exists(f)]

if missing:
    try:
        from google.colab import files
        print(f'Missing: {missing} — uploading...')
        for f in missing:
            print(f'  Select {f}:')
            uploaded = files.upload()
    except ImportError:
        print('Not in Colab — place CSV files in working directory.')
else:
    print('Both files found (from sidebar upload). Ready for next cell.')

In [ ]:
# ── Combine External Datasets with Synthetic Data ──
# Skip this cell if you only want to train on synthetic data.

import os
TARGET = 'WaitingMinutes'
external_frames = []

# --- Hospital Queue ---
hospital_path = 'hospital_engineered.csv'  # Upload to Colab first
if os.path.exists(hospital_path):
    h = pd.read_csv(hospital_path)
    h = h.rename(columns={
        'queue_length': 'QueueLength', 'people_ahead': 'PeopleAheadCallNext',
        'now_serving': 'NowServing', 'active_counters': 'ActiveCounters',
        'hour': 'HourOfDay', 'is_peak': 'IsPeakHour',
        'priority': 'CallNextPriority', 'default_avg_service_min': 'DefaultAvgServiceMinutes',
        'wait_min': TARGET,
    })
    # Safeguard 1: Filter — drop 1/2-counter disaster scenarios
    n_before = len(h)
    h = h[h['ActiveCounters'] >= 3].copy()
    h['CrossLaneQueueLength'] = h['QueueLength']
    h['ListeningCounters'] = h['ActiveCounters']
    h['BranchCode'] = 900001
    h['ServiceCode'] = 'HOSPITAL'
    h['TicketPrefix'] = 'H'
    h['EntryType'] = 1
    h['CheckedIn'] = 1
    h['EnqueueSequence'] = range(1, len(h) + 1)
    h['DayOfWeek'] = 2
    h['DataSource'] = 'hospital'
    external_frames.append(h)
    print(f'Hospital: {len(h)} rows (filtered from {n_before}, kept counters>=3)')
else:
    print(f'Hospital file not found ({hospital_path}) — skipped')

# --- College Admissions ---
college_path = 'college_engineered.csv'  # Upload to Colab first
if os.path.exists(college_path):
    c = pd.read_csv(college_path)
    c = c.rename(columns={
        'queue_length': 'QueueLength', 'people_ahead': 'PeopleAheadCallNext',
        'now_serving': 'NowServing', 'active_counters': 'ActiveCounters',
        'hour': 'HourOfDay', 'day_of_week': 'DayOfWeek',
        'is_peak': 'IsPeakHour', 'avg_service_min': 'DefaultAvgServiceMinutes',
        'wait_min': TARGET,
    })
    c['CrossLaneQueueLength'] = c['QueueLength']
    c['ListeningCounters'] = c['ActiveCounters']
    c['BranchCode'] = 900002
    c['ServiceCode'] = 'COLLEGE'
    c['TicketPrefix'] = 'C'
    c['EntryType'] = 1
    c['CheckedIn'] = 1
    c['EnqueueSequence'] = range(1, len(c) + 1)
    c['DataSource'] = 'college'
    external_frames.append(c)
    print(f'College: {len(c)} rows')
else:
    print(f'College file not found ({college_path}) — skipped')

# --- Merge ---
if external_frames:
    ext_df = pd.concat(external_frames, ignore_index=True)

    # Safeguard 2: Scale normalization — align median to synthetic
    synthetic_median = df[TARGET].dropna().median()
    for src in ext_df['DataSource'].unique():
        mask = ext_df['DataSource'] == src
        src_median = ext_df.loc[mask, TARGET].median()
        if src_median > 0:
            scale = synthetic_median / src_median
            ext_df.loc[mask, TARGET] *= scale
            ext_df['DefaultAvgServiceMinutes'] = ext_df['DefaultAvgServiceMinutes'].astype(float)
            ext_df.loc[mask, 'DefaultAvgServiceMinutes'] *= scale
            print(f'Scaled {src}: median {src_median:.1f} -> {ext_df.loc[mask, TARGET].median():.1f} min (x{scale:.2f})')

    # Cap external at 50% of synthetic (prevent domination)
    max_ext = int(len(df) * 0.5)
    if len(ext_df) > max_ext:
        ext_df = ext_df.sample(n=max_ext, random_state=42)
        print(f'Downsampled external to {max_ext} rows (50% of synthetic)')

    df = pd.concat([df, ext_df], ignore_index=True)
    print(f'\nCombined total: {len(df)} rows')
    for src in df['DataSource'].unique():
        print(f'  {src}: {(df["DataSource"] == src).sum()}')
else:
    print('No external datasets found — training on synthetic only')

## 2. Data Cleansing

In [ ]:
TARGET = 'WaitingMinutes'
n_start = len(df)

# Drop abandoned tickets (no service = no target)
df = df.dropna(subset=[TARGET])

# Drop negative/zero (system glitches)
df = df[df[TARGET] > 0].copy()

# Winsorize per source
for src in df['DataSource'].unique():
    mask = df['DataSource'] == src
    p99 = df.loc[mask, TARGET].quantile(0.99)
    p01 = df.loc[mask, TARGET].quantile(0.01)
    df.loc[mask, TARGET] = df.loc[mask, TARGET].clip(lower=p01, upper=p99)

# Dedup
df = df.drop_duplicates()

print(f'Cleaning: {n_start} -> {len(df)} rows (dropped {n_start - len(df)}, {(n_start-len(df))/n_start:.1%})')
print(f'Target range: [{df[TARGET].min():.1f}, {df[TARGET].max():.1f}] min')
print(f'Mean={df[TARGET].mean():.1f}, Median={df[TARGET].median():.1f}')

## 3. Feature Engineering

**24 features** aligned with production `MlHttpWaitTimeEstimator.BuildPayload`:
- Queue state: QueueLength, CrossLaneQueueLength, PeopleAheadCallNext, NowServing
- Capacity: ActiveCounters, ListeningCounters
- Time: HourOfDay, DayOfWeek, IsPeakHour
- Service: DefaultAvgServiceMinutes
- Entry: EnqueueSequence, EntryType, CheckedIn
- Slot (NaN for walk-ins): SlotActive, CallNextPriority, SlotDurationMinutes, MinutesUntilSlotEnd, MinutesSinceSlotStart, OnlineSlotsPerSlot, OnlineBookedInSlot, WalkInInSlot
- Identity: BranchCode, ServiceCode, TicketPrefix

In [ ]:
FEATURE_NUMERIC = [
    'QueueLength', 'CrossLaneQueueLength', 'PeopleAheadCallNext', 'NowServing',
    'ActiveCounters', 'ListeningCounters',
    'HourOfDay', 'DayOfWeek', 'IsPeakHour',
    'DefaultAvgServiceMinutes',
    'EnqueueSequence', 'EntryType', 'CheckedIn',
    'SlotActive', 'CallNextPriority', 'SlotDurationMinutes',
    'MinutesUntilSlotEnd', 'MinutesSinceSlotStart',
    'OnlineSlotsPerSlot', 'OnlineBookedInSlot', 'WalkInInSlot',
    'BranchCode',
]
FEATURE_CAT = ['ServiceCode', 'TicketPrefix']
ALL_FEATURES = FEATURE_NUMERIC + FEATURE_CAT

# Ensure columns exist
for col in FEATURE_NUMERIC:
    if col not in df.columns: df[col] = np.nan
    df[col] = pd.to_numeric(df[col], errors='coerce')
for col in FEATURE_CAT:
    if col not in df.columns: df[col] = 'UNKNOWN'

# Report NaN (expected for slot features on walk-ins)
nan_counts = df[FEATURE_NUMERIC].isna().sum()
nan_feats = nan_counts[nan_counts > 0]
print(f'{len(ALL_FEATURES)} features ({len(FEATURE_NUMERIC)} numeric + {len(FEATURE_CAT)} categorical)')
if len(nan_feats) > 0:
    print('\nNaN features (handled natively by HistGBR):')
    for f, c in nan_feats.items():
        print(f'  {f}: {c} ({c/len(df):.1%})')

## 4. Model Training

- **HistGradientBoostingRegressor** with Poisson loss (natural for non-negative, right-skewed wait times)
- **Temporal split** (train on earlier data, test on later) to prevent data leakage
- **No imputation needed** — HistGBR handles NaN natively for slot features

In [ ]:
# Per-source temporal split (same as train_wait_model.py)
# Each DataSource is split independently to prevent all external data
# ending up on one side (this bug caused MAE=468 before we fixed it).

sort_col = 'EnqueueSequence'
test_ratio = 0.2

if 'DataSource' not in df.columns:
    df['DataSource'] = 'synthetic'

train_parts, test_parts = [], []
for source, group in df.groupby('DataSource'):
    group = group.sort_values(sort_col)
    split_idx = int(len(group) * (1 - test_ratio))
    train_parts.append(group.iloc[:split_idx])
    test_parts.append(group.iloc[split_idx:])

train_df = pd.concat(train_parts, ignore_index=True)
test_df  = pd.concat(test_parts, ignore_index=True)

X_train, y_train = train_df[ALL_FEATURES], train_df[TARGET].astype(float)
X_test,  y_test  = test_df[ALL_FEATURES],  test_df[TARGET].astype(float)

print(f'Per-source temporal split: train={len(train_df)}, test={len(test_df)} ({test_ratio:.0%} test)')
for src in df['DataSource'].unique():
    n_tr = (train_df['DataSource'] == src).sum()
    n_te = (test_df['DataSource'] == src).sum()
    print(f'  {src}: train={n_tr}, test={n_te}')
print(f'Train wait: mean={y_train.mean():.1f}, Test wait: mean={y_test.mean():.1f}')

In [ ]:
%%time
# Build and train model
preprocessor = ColumnTransformer([
    ('num', 'passthrough', FEATURE_NUMERIC),
    ('cat', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1), FEATURE_CAT),
])

pipeline = Pipeline([
    ('preprocess', preprocessor),
    ('model', HistGradientBoostingRegressor(
        loss='poisson',
        max_depth=6, learning_rate=0.08, max_iter=300,
        min_samples_leaf=8, random_state=42,
    )),
])

# Safeguard 3: Sample weighting — synthetic/primary=1.0, external=0.1
# This ensures external data adds signal without dominating the loss function.
train_weights = train_df['DataSource'].map(
    lambda s: 1.0 if s in ('synthetic', 'primary', 'sql') else 0.1
).values
pipeline.fit(X_train, y_train, model__sample_weight=train_weights)

weight_summary = train_df['DataSource'].value_counts()
print('Model trained with sample weighting:')
for src in weight_summary.index:
    w = 1.0 if src in ('synthetic', 'primary', 'sql') else 0.1
    print(f'  {src}: {weight_summary[src]} rows x weight {w}')

## 5. Evaluation

In [ ]:
# ML predictions
ml_pred = np.maximum(0, pipeline.predict(X_test))

# Formula baseline (matches production WaitTimeEstimator.cs)
def formula_baseline(row):
    ahead = max(0, row.get('PeopleAheadCallNext', 0) or 0)
    now_s = max(0, row.get('NowServing', 0) or 0)
    avg = row.get('DefaultAvgServiceMinutes', 10) or 10
    counters = max(1, row.get('ListeningCounters', 1) or 1)
    return max(0, (ahead + now_s) * avg / counters)

fm_pred = test_df.apply(formula_baseline, axis=1).values

# Metrics
ml_mae  = mean_absolute_error(y_test, ml_pred)
ml_rmse = np.sqrt(mean_squared_error(y_test, ml_pred))
ml_r2   = r2_score(y_test, ml_pred)
fm_mae  = mean_absolute_error(y_test, fm_pred)
fm_rmse = np.sqrt(mean_squared_error(y_test, fm_pred))
fm_r2   = r2_score(y_test, fm_pred)

# Cross-validation (on full dataset)
# FIX: Sort by DataSource → BranchCode → EnqueueSequence so TimeSeriesSplit
# respects temporal order within each branch. Previously sorted only by
# EnqueueSequence, which is independently numbered per branch/source —
# mixing them destroyed the physical time ordering.
cv_sort_cols = ['DataSource', 'BranchCode', 'EnqueueSequence']
X_all = df.sort_values(cv_sort_cols)[ALL_FEATURES]
y_all = df.sort_values(cv_sort_cols)[TARGET].astype(float)
tscv = TimeSeriesSplit(n_splits=5)
cv_mae = -cross_val_score(pipeline, X_all, y_all, cv=tscv, scoring='neg_mean_absolute_error')

print('=' * 60)
print('  RESULTS')
print('=' * 60)
print(f'  ML Model:     MAE = {ml_mae:.2f} min   RMSE = {ml_rmse:.2f}   R2 = {ml_r2:.4f}')
print(f'  Formula:      MAE = {fm_mae:.2f} min   RMSE = {fm_rmse:.2f}   R2 = {fm_r2:.4f}')
print(f'  CV MAE:       {np.mean(cv_mae):.2f} min (5-fold TimeSeriesSplit)')
print(f'  Improvement:  ML is {fm_mae/ml_mae:.1f}x more accurate than formula')

In [ ]:
# Feature Importance
from sklearn.inspection import permutation_importance

perm = permutation_importance(pipeline, X_test, y_test, n_repeats=5, random_state=42, n_jobs=-1)
imp = sorted(zip(ALL_FEATURES, perm.importances_mean), key=lambda x: -x[1])

fig, ax = plt.subplots(figsize=(10, 6))
names, vals = zip(*imp[:15])
ax.barh(range(len(names)), vals, color='steelblue')
ax.set_yticks(range(len(names)))
ax.set_yticklabels(names)
ax.invert_yaxis()
ax.set_xlabel('Permutation Importance')
ax.set_title('Top 15 Feature Importance')
plt.tight_layout()
plt.show()

In [ ]:
# Predicted vs Actual + Residuals
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Predicted vs Actual
ax = axes[0]
ax.scatter(y_test, ml_pred, s=4, alpha=0.3, color='steelblue', label='ML')
ax.scatter(y_test, fm_pred, s=4, alpha=0.15, color='orange', label='Formula')
lim = [0, max(y_test.max(), ml_pred.max()) * 1.05]
ax.plot(lim, lim, 'r--', lw=1, label='Perfect')
ax.set_xlim(lim); ax.set_ylim(lim)
ax.set_xlabel('Actual (min)'); ax.set_ylabel('Predicted (min)')
ax.set_title('Predicted vs Actual')
ax.legend(markerscale=4)

# Residual histogram
ax2 = axes[1]
ml_resid = ml_pred - y_test.values
fm_resid = fm_pred - y_test.values
rng_val = max(abs(ml_resid).max(), 30)
ax2.hist(ml_resid, bins=50, range=(-rng_val, rng_val), alpha=0.6, color='steelblue', label=f'ML (MAE={ml_mae:.1f})')
ax2.hist(fm_resid, bins=50, range=(-rng_val, rng_val), alpha=0.4, color='orange', label=f'Formula (MAE={fm_mae:.1f})')
ax2.axvline(0, color='red', ls='--')
ax2.set_xlabel('Residual (min)'); ax2.set_title('Residual Distribution')
ax2.legend()

# Error by service type
ax3 = axes[2]
test_df_copy = test_df.copy()
test_df_copy['ml_error'] = np.abs(ml_pred - y_test.values)
test_df_copy['fm_error'] = np.abs(fm_pred - y_test.values)
svc_ml = test_df_copy.groupby('ServiceCode')['ml_error'].mean().sort_values(ascending=False)
svc_fm = test_df_copy.groupby('ServiceCode')['fm_error'].mean()
x_pos = range(len(svc_ml))
ax3.bar([p - 0.2 for p in x_pos], svc_ml.values, 0.4, color='steelblue', label='ML')
ax3.bar([p + 0.2 for p in x_pos], [svc_fm.get(s, 0) for s in svc_ml.index], 0.4, color='orange', label='Formula')
ax3.set_xticks(list(x_pos))
ax3.set_xticklabels(svc_ml.index, rotation=45, ha='right', fontsize=8)
ax3.set_ylabel('MAE (min)'); ax3.set_title('MAE by Service Type')
ax3.legend()

plt.suptitle('QMS Wait-Time Model Evaluation', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## Summary

| Metric | ML Model | Formula Baseline |
|--------|----------|------------------|
| MAE    | ~11-13 min | ~20-22 min |
| R²     | ~0.60-0.64 | < 0 (worse than mean) |
| Improvement | **1.7-1.8x** more accurate | baseline |

### Architecture
```
Customer takes ticket
    → WaitTimeFeatureBuilder computes 24 features
    → MlHttpWaitTimeEstimator sends to Python sidecar
    → HistGradientBoosting predicts wait time
    → Sanity clamp (50% formula ≤ ML ≤ 150% simulation)
    → Display on ticket / app
```

### Deployment Phases
1. **Pre-launch**: Train on synthetic data (this notebook)
2. **Post-launch**: Retrain on real `ML_TRAINING_DATA` from system
3. **Mature**: Full real data, consider Two-Stage model for anomalies

In [ ]:
import joblib, json
from pathlib import Path

MODEL_PATH = Path('wait_model.joblib')
METRICS_PATH = Path('model_metrics.json')

# ── Export model bundle (same format as train_wait_model.py) ──
joblib.dump({
    'pipeline': pipeline,
    'feature_numeric': FEATURE_NUMERIC,
    'feature_categorical': FEATURE_CAT,
    'target': TARGET,
    'version': 'v2',
}, MODEL_PATH)

# ── Export metrics ──
metrics = {
    'data_source': 'notebook (synthetic)',
    'row_count': len(df),
    'test_ml': {'mae_minutes': round(ml_mae, 3), 'rmse_minutes': round(ml_rmse, 3), 'r2': round(ml_r2, 4)},
    'test_formula_baseline': {'mae_minutes': round(fm_mae, 3), 'rmse_minutes': round(fm_rmse, 3), 'r2': round(fm_r2, 4)},
    'cv_mae_mean': round(float(np.mean(cv_mae)), 3),
    'feature_count': len(ALL_FEATURES),
    'features_numeric': FEATURE_NUMERIC,
    'features_categorical': FEATURE_CAT,
}
METRICS_PATH.write_text(json.dumps(metrics, indent=2))

print(f'Saved: {MODEL_PATH} ({MODEL_PATH.stat().st_size / 1024:.0f} KB)')
print(f'Saved: {METRICS_PATH}')
print(f'\nDeploy: copy wait_model.joblib to ml/ then restart serve_wait_model.py')